# Baseline: Qwen3-ASR
## Vietnamese-English Code-Switching ASR
Model: `Qwen/Qwen3-ASR-0.6B` hoặc `Qwen/Qwen3-ASR-1.7B`

In [ ]:
!pip install -U qwen-asr
!pip install -q evaluate==0.4.1 jiwer==3.0.3 librosa==0.10.1

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import torch
import librosa
from tqdm.notebook import tqdm
from qwen_asr import Qwen3ASRModel
import evaluate
from jiwer import wer

os.environ["WANDB_DISABLED"] = "true"

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_ID      = "Qwen/Qwen3-ASR-0.6B"  # đổi sang 1.7B nếu VRAM đủ
AUDIO_DIR     = "./test/audio"
TRANSCRIPT    = "./test/transcript.json"
OUTPUT_CSV    = "./results_qwen3_asr.csv"
SAMPLING_RATE = 16000
BATCH_SIZE    = 8

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def compute_ser(refs, hyps):
    return sum(r.strip() != h.strip() for r, h in zip(refs, hyps)) / len(refs) * 100

def compute_all_metrics(refs, hyps):
    wer_m = evaluate.load("wer")
    cer_m = evaluate.load("cer")
    refs_n = [normalize_text(r) for r in refs]
    hyps_n = [normalize_text(h) for h in hyps]
    return {
        "wer_raw":  round(wer_m.compute(references=refs,   predictions=hyps)   * 100, 2),
        "cer_raw":  round(cer_m.compute(references=refs,   predictions=hyps)   * 100, 2),
        "ser_raw":  round(compute_ser(refs, hyps), 2),
        "wer_norm": round(wer_m.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "cer_norm": round(cer_m.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "ser_norm": round(compute_ser(refs_n, hyps_n), 2),
    }

def load_test_samples(audio_dir, transcript_path):
    with open(transcript_path, encoding="utf-8") as f:
        transcripts = json.load(f)
    samples, missing = [], 0
    for fname, meta in transcripts.items():
        path = os.path.join(audio_dir, fname)
        if os.path.exists(path):
            samples.append({"filename": fname, "audio_path": path, "text": meta["text"]})
        else:
            missing += 1
    print(f"Loaded {len(samples)} samples (skipped {missing} missing)")
    return samples

print("Helpers defined.")

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID}...")
model = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    max_inference_batch_size=BATCH_SIZE,
    max_new_tokens=256,
)
print("Model ready.")

In [ ]:
# ── Load test data ─────────────────────────────────────────────────────────
test_samples = load_test_samples(AUDIO_DIR, TRANSCRIPT)

In [ ]:
# ── Inference ──────────────────────────────────────────────────────────────
filenames, hypotheses, references = [], [], []

for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Inference"):
    batch       = test_samples[i : i + BATCH_SIZE]
    audio_paths = [s["audio_path"] for s in batch]

    results = model.transcribe(audio=audio_paths, language="Vietnamese")

    hypotheses.extend([r.text for r in results])
    filenames.extend([s["filename"] for s in batch])
    references.extend([s["text"] for s in batch])

print(f"Done — {len(hypotheses)} samples")

In [ ]:
# ── Metrics & save ─────────────────────────────────────────────────────────
metrics = compute_all_metrics(references, hypotheses)
print(f"Before norm → WER: {metrics['wer_raw']}% | CER: {metrics['cer_raw']}% | SER: {metrics['ser_raw']}%")
print(f"After  norm → WER: {metrics['wer_norm']}% | CER: {metrics['cer_norm']}% | SER: {metrics['ser_norm']}%")

df = pd.DataFrame({"filename": filenames, "reference": references, "hypothesis": hypotheses})
df["wer_per_sentence"] = df.apply(
    lambda r: wer(normalize_text(r["reference"]), normalize_text(r["hypothesis"])) * 100, axis=1
)
df.sort_values("wer_per_sentence", ascending=False).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")